# ADMET · Metabolism assessment for rapamycin and its rapalogs

Fourth in the series (potency → **A** → **D** → **M**etabolism). Same rapalogs molecules throughout.
Claims are grounded in primary literature (PubMed) and cited.

## Purpose

**Metabolism** asks: *how does the body chemically break the drug down* — which enzymes act,
how fast, and does the drug inhibit those enzymes (a drug–drug-interaction, DDI, risk)? 

## Conclusion
For the rapalogs the answer is dominated by **CYP3A4/3A5**, with **P-glycoprotein** efflux in the gut.

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import io, zipfile, requests
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, rdFingerprintGenerator
from chembl_webresource_client.new_client import new_client
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, r2_score

# Notebook is stripped before push; each step's result is saved as a PNG here.
RESULTS = Path("../result_rapamycin_ADMET_Metabolism"); RESULTS.mkdir(exist_ok=True)
def save_fig(fig, name): fig.savefig(RESULTS / name, dpi=150, bbox_inches="tight")
def save_img(obj, name):
    p = RESULTS / name
    obj.save(p) if hasattr(obj, "save") else p.write_bytes(obj.data)
def save_df_png(df, name, title=None):
    fig, ax = plt.subplots(figsize=(min(2 + len(df.columns) * 1.7, 18), 0.7 + 0.45 * len(df))); ax.axis("off")
    if title: ax.set_title(title, fontsize=10, loc="left")
    t = ax.table(cellText=df.astype(str).values, colLabels=list(df.columns), loc="center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1, 1.4); t.auto_set_column_width(range(len(df.columns)))
    fig.savefig(RESULTS / name, dpi=150, bbox_inches="tight"); plt.close(fig)

mol_client = new_client.molecule
RAPALOGS = {"Sirolimus (rapamycin)": "CHEMBL413", "Everolimus": "CHEMBL1908360",
            "Temsirolimus": "CHEMBL1201182", "Ridaforolimus": "CHEMBL2103839",
            "Zotarolimus": "CHEMBL219410"}
smiles = {n: mol_client.filter(molecule_chembl_id=c).only(["molecule_structures"])[0]
          ["molecule_structures"]["canonical_smiles"] for n, c in RAPALOGS.items()}
mols = {n: Chem.MolFromSmiles(s) for n, s in smiles.items()}

# Therapeutics Data Commons 'admet_group' benchmark (Huang et al. 2022), cached (git-ignored)
ZIP = Path("../data/tdc_admet_group.zip")
if not ZIP.exists():
    ZIP.write_bytes(requests.get("https://dataverse.harvard.edu/api/access/datafile/4426004",
                                 headers={"User-Agent": "Mozilla/5.0"}, timeout=120).content)
_zf = zipfile.ZipFile(ZIP)
_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def _bv(s):
    m = Chem.MolFromSmiles(s); return _GEN.GetFingerprint(m) if m else None
def _arr(bv):
    a = np.zeros((2048,), np.uint8); DataStructs.ConvertToNumpyArray(bv, a); return a
def _load(name):
    d = pd.concat([pd.read_csv(_zf.open(f"admet_group/{name}/{f}")) for f in ("train_val.csv", "test.csv")], ignore_index=True)
    return d[["Drug", "Y"]].dropna()
rap_bv = {n: _GEN.GetFingerprint(m) for n, m in mols.items()}

def run_models(models, pred_png, ad_png):
    "models: list of (label, tdc_dataset, task in clf/reg). Trains RF, predicts rapalogs + applicability domain."
    rows = {n.split(" ")[0]: {"rapalog": n.split(" ")[0]} for n in rap_bv}; cols = []
    for label, ds, task in models:
        d = _load(ds); bvs = [_bv(s) for s in d["Drug"]]
        keep = [i for i, b in enumerate(bvs) if b is not None]; bvs = [bvs[i] for i in keep]
        X = np.array([_arr(b) for b in bvs]); y = d["Y"].to_numpy()[keep]
        if task == "clf":
            m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
            proba = cross_val_predict(m, X, y, cv=5, method="predict_proba", n_jobs=-1)[:, 1]
            metric = f"ROC-AUC={roc_auc_score(y, proba):.2f}"; m.fit(X, y); pr = lambda a: round(m.predict_proba(a)[0, 1], 2)
        else:
            m = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
            p = cross_val_predict(m, X, y, cv=5, n_jobs=-1); metric = f"R2={r2_score(y, p):.2f}"; m.fit(X, y); pr = lambda a: round(float(m.predict(a)[0]), 2)
        print(f"{label:26s} ({ds:28s}) n={len(y):5d}  CV {metric}")
        cols.append(label)
        for n, bv in rap_bv.items():
            k = n.split(" ")[0]
            rows[k][label] = pr(_arr(bv).reshape(1, -1))
            rows[k][label + " AD"] = round(max(DataStructs.BulkTanimotoSimilarity(bv, bvs)), 2)
    df = pd.DataFrame(rows.values())
    save_df_png(df, pred_png, "de-novo ML predictions + applicability domain (max Tanimoto to training set)")
    fig, ax = plt.subplots(figsize=(9.5, 4)); x = np.arange(len(df)); w = 0.8 / len(cols)
    for i, label in enumerate(cols):
        ax.bar(x + (i - (len(cols) - 1) / 2) * w, df[label + " AD"], w, label=label.split(" ")[0][:16])
    ax.axhline(0.3, color="red", ls="--", label="in-domain ~0.3")
    ax.set_xticks(x); ax.set_xticklabels(df["rapalog"], rotation=20)
    ax.set(ylabel="max Tanimoto to training set", ylim=(0, 1.05),
           title="Applicability domain of de-novo ML models for the rapalogs")
    ax.legend(fontsize=7, ncol=2); plt.tight_layout(); save_fig(fig, ad_png); plt.show()
    return df

print("Setup ready.")

## Step 1 — Honest caveat first (Yeah I need different molecules to try QSAR)

**The 900 Da problem, again.** Metabolism QSARs (CYP inhibition/substrate, clearance) are trained
on small drug-like molecules; for ~900 Da macrocycles, structure-only predictions are unreliable and the **applicability domain** is the honest verdict.

**But metabolism is the one axis where *in silico* may partly connect** — because the rapalogs'
metabolic fate is well characterised: they are **CYP3A4/3A5 substrates** and **P-gp substrates**,
undergoing extensive hepatic and gut-wall CYP3A metabolism, which drives major DDIs with CYP3A
inhibitors/inducers (Paine et al., 2004; MacDonald et al., 2000; Kirchner et al., 2004). Note that
CYP3A4 *substrate* status (how they're cleared) differs from CYP3A4 *inhibition* (a DDI liability we
can screen); we run the inhibition/clearance models below and read them against the measured facts.

In [ ]:
tbl = pd.DataFrame({"MW (Da)": {n: round(Descriptors.MolWt(m)) for n, m in mols.items()},
                    "cLogP": {n: round(Descriptors.MolLogP(m), 1) for n, m in mols.items()}})
print(tbl.to_string())
save_df_png(tbl.reset_index().rename(columns={"index": "drug"}), "step1_properties.png", "Step 1 - the molecules")
grid = Draw.MolsToGridImage(list(mols.values()), legends=list(mols.keys()), molsPerRow=3, subImgSize=(300, 230))
save_img(grid, "step1_structures.png"); grid

## Step 2 — A rigorous in-silico metabolism stack (methods; not run here)

| Endpoint | In-silico model | Trustworthy for rapalogs? | Reference / data |
|---|---|---|---|
| **CYP inhibition** (3A4, 2C9, 2D6) | ML classifier on qHTS libraries | ⚠️ rapalogs out of domain, but CYP3A4 is the right enzyme | Veith et al., 2009 (data); TDC (Huang et al., 2022) |
| **CYP substrate** (who metabolizes it) | ML classifier / site-of-metabolism | ⚠️ rapalogs are known CYP3A4 substrates (measured) | measured; ML small-molecule-trained |
| **Metabolic (hepatic) clearance** | QSAR / ML regression | ❌ out of domain; low CL under-captured | Gombar & Hall, 2013; Obach et al., 2008 |
| **Gut-wall P-gp efflux** | P-gp substrate ML | (measured substrate) | Paine et al., 2004 |

> **Dominant metabolic mode: CYP3A4/3A5 (hepatic + gut) with P-gp efflux.** This is a *mechanistic*
> fact from wet-lab studies; structure-only ML can screen CYP-inhibition liability but does not
> establish the substrate/clearance behaviour that actually governs rapalog exposure.

**What we run in Step 4:** CYP3A4 & CYP2D6 **inhibition** classifiers and a **hepatocyte clearance**
regressor (TDC data), predicted for the rapalogs with applicability domain.

In [ ]:
measured = pd.DataFrame([
    {"drug": "Sirolimus", "primary_enzyme": "CYP3A4/3A5 (hepatic + gut)", "transporter": "P-gp substrate",
     "key_metabolites": "hydroxy-/demethyl-sirolimus (>7 metabolites)", "DDI": "strong with CYP3A/P-gp inhibitors/inducers",
     "ref": "MacDonald 2000; Mahalati 2001; Paine 2004"},
    {"drug": "Everolimus", "primary_enzyme": "CYP3A4 (also 3A5, 2C8)", "transporter": "P-gp substrate",
     "key_metabolites": "hydroxy/demethyl metabolites", "DDI": "CYP3A/P-gp mediated", "ref": "Kirchner 2004"},
    {"drug": "Temsirolimus", "primary_enzyme": "CYP3A4 (prodrug -> sirolimus)", "transporter": "P-gp substrate",
     "key_metabolites": "sirolimus (active) + others", "DDI": "CYP3A mediated", "ref": "class / label"},
    {"drug": "Ridaforolimus", "primary_enzyme": "CYP3A4", "transporter": "P-gp substrate",
     "key_metabolites": "CYP3A metabolites", "DDI": "CYP3A mediated", "ref": "class / label"},
    {"drug": "Zotarolimus", "primary_enzyme": "CYP3A4", "transporter": "P-gp substrate",
     "key_metabolites": "CYP3A metabolites", "DDI": "CYP3A mediated", "ref": "class / label"},
])
save_df_png(measured, "step3_measured_metabolism.png", "Step 3 - measured metabolism")
measured

## Step 3 — Measured metabolism (above) and Step 4 — run the ML stack

**Measured summary:** the rapalogs are uniformly **CYP3A4/3A5 substrates and P-gp substrates**;
hepatic + intestinal CYP3A metabolism dominates and is the source of their extensive drug–drug
interactions. Now we run the structure-only ML metabolism models and read them against this.

In [ ]:
ml = run_models([("CYP3A4 inhibition P", "cyp3a4_veith", "clf"),
                 ("CYP2D6 inhibition P", "cyp2d6_veith", "clf"),
                 ("Hepatocyte CL (log)", "clearance_hepatocyte_az", "reg")],
                "step4_ml_metabolism_predictions.png", "step4_applicability_domain.png")
ml

### What the run shows, and caveats (Step 5)

- **CYP3A4-inhibition model** predicts the rapalogs are **not** strong CYP3A4 inhibitors (P ≈ 0.3),
  consistent with their being CYP3A4 *substrates* rather than inhibitors — but the **applicability
  domain is low (~0.25)**, so this is a low-confidence, out-of-domain read. Critically, a "not an
  inhibitor" verdict says nothing about the *substrate*/clearance behaviour that actually governs
  rapalog metabolism.
- **CYP2D6** (an enzyme the rapalogs do not rely on) is predicted low — a sensible negative.
- **Hepatocyte clearance** is barely learnable here (low cross-validated R²); the rapalogs' true
  clearance is metabolism-limited (CYP3A4) and best measured.

**Caveats.** (i) 900 Da → structure-only ML is extrapolation; the low AD column is the verdict.
(ii) The decisive metabolic facts — CYP3A4/3A5 substrate status, P-gp efflux, metabolites and DDIs —
come from **measured** studies, not de-novo prediction. (iii) **Inhibition ≠ substrate**: the models
screen DDI/inhibition liability, not the clearance route that governs exposure. **Measured data is
decisive.**

## Step 6 — Citations (ACS style)

Bibliographic metadata retrieved from **PubMed**.

1. Paine, M. F.; Leung, L. Y.; Watkins, P. B. New Insights into Drug Absorption: Studies with
   Sirolimus. *Ther. Drug Monit.* **2004**, *26* (5), 463–467.
   https://doi.org/10.1097/00007691-200410000-00001.
2. MacDonald, A.; Scarola, J.; Burke, J. T.; Zimmerman, J. J. Clinical Pharmacokinetics and
   Therapeutic Drug Monitoring of Sirolimus. *Clin. Ther.* **2000**, *22* (Suppl. B), B101–B121.
   https://doi.org/10.1016/S0149-2918(00)89027-X.
3. Mahalati, K.; Kahan, B. D. Clinical Pharmacokinetics of Sirolimus. *Clin. Pharmacokinet.*
   **2001**, *40* (8), 573–585. https://doi.org/10.2165/00003088-200140080-00002.
4. Kirchner, G. I.; Meier-Wiedenbach, I.; Manns, M. P. Clinical Pharmacokinetics of Everolimus.
   *Clin. Pharmacokinet.* **2004**, *43* (2), 83–95. https://doi.org/10.2165/00003088-200443020-00002.
5. Veith, H.; Southall, N.; Huang, R.; James, T.; Fayne, D.; Artemenko, N.; Shen, M.; Inglese, J.;
   Austin, C. P.; Lloyd, D. G.; Auld, D. S. Comprehensive Characterization of Cytochrome P450
   Isozyme Selectivity across Chemical Libraries. *Nat. Biotechnol.* **2009**, *27* (11), 1050–1055.
   https://doi.org/10.1038/nbt.1581.
6. Gombar, V. K.; Hall, S. D. Quantitative Structure–Activity Relationship Models of Clinical
   Pharmacokinetics: Clearance and Volume of Distribution. *J. Chem. Inf. Model.* **2013**, *53* (4),
   948–957. https://doi.org/10.1021/ci400001u.
7. Obach, R. S.; Lombardo, F.; Waters, N. J. Trend Analysis of a Database of Intravenous
   Pharmacokinetic Parameters in Humans for 670 Drug Compounds. *Drug Metab. Dispos.* **2008**,
   *36* (7), 1385–1405. https://doi.org/10.1124/dmd.108.020479.
8. Huang, K.; Fu, T.; Gao, W.; Zhao, Y.; Roohani, Y.; Leskovec, J.; Coley, C. W.; Xiao, C.; Sun, J.;
   Zitnik, M. Artificial Intelligence Foundation for Therapeutic Science. *Nat. Chem. Biol.* **2022**,
   *18* (10), 1033–1036. https://doi.org/10.1038/s41589-022-01131-2.

*Attribution: PubMed. Training data are the TDC `admet_group` benchmark (cyp3a4_veith, cyp2d6_veith,
clearance_hepatocyte_az).*